In [1]:
import pandas as pd

In [2]:
# Load yesterday's generated CSV file into memory
df_events = pd.read_csv("data/product_events.csv")

# Print the size and show a 5-row table view
print("Dataset loaded successfully! Total records:", len(df_events))
df_events.head()


Dataset loaded successfully! Total records: 20000


,event_id,user_id,product_id,category,event_type,price,quantity,revenue,device,country,session_id,event_time
0,e_002263,u_00318,p_0001,apparel,checkout,17.38,NaN,0.0,desktop,US,s_002247,2026-01-01T00:05:18+00:00
1,e_006995,u_00449,p_0042,home,page_view,127.08,NaN,0.0,mobile,GB,s_004450,2026-01-01T00:39:05+00:00
2,e_008380,u_00227,p_0040,grocery,product_view,200.84,NaN,0.0,tablet,FR,s_001606,2026-01-01T00:43:35+00:00
3,e_012118,u_00312,p_0066,grocery,page_view,14.64,NaN,0.0,tablet,CA,s_001390,2026-01-01T00:44:25+00:00
4,e_001424,u_00279,p_0039,books,product_view,110.25,NaN,0.0,desktop,JP,s_005392,2026-01-01T00:47:51+00:00


In [3]:
def calculate_conversion_rate(total_page_views, total_checkouts):
    """Calculates the percentage of page views that converted into checkouts."""
    if total_page_views == 0:
        return 0.0

    conversion_rate = (total_checkouts / total_page_views) * 100
    return f"{conversion_rate:.2f}%"


# Let's count our actual dataset rows to test it!
# (We filter rows where event_type matches specific actions)
views = len(df_events[df_events["event_type"] == "page_view"])
checkouts = len(df_events[df_events["event_type"] == "checkout"])

# Run the function using real data
final_metric = calculate_conversion_rate(views, checkouts)
print(f"Real Web Conversion Rate: {final_metric}")


Real Web Conversion Rate: 26.99%


In [4]:
def get_high_value_events(dataframe, min_revenue):
    """Filters the dataset to return only events that generated revenue above the minimum threshold."""
    # Filter the dataframe rows based on the revenue column condition
    filtered_df = dataframe[dataframe["revenue"] > min_revenue]

    # Return the new filtered dataframe
    return filtered_df


# --- Run the function using your live dataset ---
# Let's find all interactions that brought in more than $100.00
high_value_data = get_high_value_events(df_events, 100.00)

print(f"Successfully isolated {len(high_value_data)} high-value transactions!")
high_value_data.head()


Successfully isolated 2162 high-value transactions!


,event_id,user_id,product_id,category,event_type,price,quantity,revenue,device,country,session_id,event_time
6,e_015898,u_00222,p_0068,sports,purchase,123.53,3.0,370.59,tablet,JP,s_001969,2026-01-01T00:55:25+00:00
14,e_012311,u_00388,p_0073,grocery,purchase,473.69,3.0,1421.07,tablet,DE,s_002501,2026-01-01T03:31:50+00:00
29,e_018645,u_00299,p_0076,beauty,purchase,357.91,3.0,1073.73,tablet,FR,s_002763,2026-01-01T07:28:05+00:00
34,e_017678,u_00277,p_0063,beauty,purchase,80.66,2.0,161.32,desktop,FR,s_003546,2026-01-01T09:43:53+00:00
38,e_014907,u_00037,p_0067,books,purchase,439.97,3.0,1319.91,tablet,DE,s_001403,2026-01-01T10:54:17+00:00


In [5]:
def categorize_device_vibe(device_name):
    """Categorizes user interaction styles based on their hardware platform type."""
    # Ensure the string is clean and lowercase to prevent matching bugs
    device_clean = str(device_name).lower().strip()

    if device_clean in ["mobile", "tablet"]:
        return "On-The-Go"
    elif device_clean == "desktop":
        return "Stationary Workspace"
    else:
        return "Unknown Platform"


# --- Test the function across your actual dataset rows ---
# We apply the function to the 'device' column to create a new behavioral insights column
df_events["user_vibe"] = df_events["device"].apply(categorize_device_vibe)

# Print a breakdown count of our new categories to prove it worked
print(df_events["user_vibe"].value_counts())


user_vibe
On-The-Go               13462
Stationary Workspace     6538
Name: count, dtype: int64


In [6]:
def format_currency_report(amount):
    """Formats raw numeric floats into a clean, standardized currency string."""
    try:
        # Force convert the input to a float to ensure numeric handling
        value = float(amount)
        return f"${value:,.2f}"
    except (ValueError, TypeError):
        # Prevent app crashes if bad data or missing values are passed in
        return "$0.00"


# --- Test the function across your actual dataset columns ---
# We calculate the total revenue of our high-value dataset from earlier
total_raw_revenue = df_events["revenue"].sum()

# Clean it up using our new function
formatted_revenue = format_currency_report(total_raw_revenue)

print("Raw Total Summed Value:", total_raw_revenue)
print("Formatted Financial Report Value:", formatted_revenue)


Raw Total Summed Value: 1294374.58
Formatted Financial Report Value: $1,294,374.58


In [7]:
def get_time_of_day_bucket(timestamp_string):
    """Parses a timestamp string and returns the corresponding time-of-day bucket."""
    try:
        # Convert the raw text timestamp into a real Python datetime object
        dt = pd.to_datetime(timestamp_string)
        hour = dt.hour
        
        if 5 <= hour < 12:
            return "Morning"
        elif 12 <= hour < 17:
            return "Afternoon"
        elif 17 <= hour < 22:
            return "Evening"
        else:
            return "Night"
    except Exception:
        # Return a safe fallback if the timestamp string is corrupt or missing
        return "Unknown"

# --- Test the function across your dataset ---
# Apply the function to your timestamp column to see when users are active
df_events["time_of_day"] = df_events["event_time"].apply(get_time_of_day_bucket)

# Print out the results summary
print(df_events["time_of_day"].value_counts())


time_of_day
Night        6003
Morning      5747
Afternoon    4210
Evening      4040
Name: count, dtype: int64


In [11]:
import psycopg
import pandas as pd

# 1. Establish connection to your PostgreSQL database
conn_str = "dbname=product_events_demo user=audreyzhong host=localhost"

# 2. Write your first analytical SQL Query
sql_query_1 = """
SELECT device, COUNT(*) as total_events 
FROM product_events 
GROUP BY device 
ORDER BY total_events DESC;
"""

# 3. Execute the query and pull it into a notebook view
with psycopg.connect(conn_str) as conn:
    df_sql_result = pd.read_sql_query(sql_query_1, conn)

# Display the results
print("SQL Query 1 Output:")
print(df_sql_result)


SQL Query 1 Output:
    device  total_events
0   mobile          6769
1   tablet          6693
2  desktop          6538


/var/folders/7n/1gz7py493ldd6nlskw8mzbdc0000gn/T/ipykernel_72580/2240536282.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sql_result = pd.read_sql_query(sql_query_1, conn)


In [9]:
!pip install psycopg[binary]


zsh:1: no matches found: psycopg[binary]


In [10]:
!pip install 'psycopg[binary]'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 4.8 MB/s  0:00:01m 5.1 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [psycopg]


In [12]:
sql_query_2 = """
SELECT product_id, SUM(revenue) as total_revenue
FROM product_events
WHERE event_type = 'checkout'
GROUP BY product_id
ORDER BY total_revenue DESC
LIMIT 5;
"""

with psycopg.connect(conn_str) as conn:
    df_sql_result = pd.read_sql_query(sql_query_2, conn)

print("SQL Query 2 Output (Top 5 Products by Revenue):")
print(df_sql_result)


SQL Query 2 Output (Top 5 Products by Revenue):
  product_id  total_revenue
0     p_0059            0.0
1     p_0002            0.0
2     p_0047            0.0
3     p_0065            0.0
4     p_0009            0.0


/var/folders/7n/1gz7py493ldd6nlskw8mzbdc0000gn/T/ipykernel_72580/2247715202.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sql_result = pd.read_sql_query(sql_query_2, conn)


In [13]:
diagnostic_query = "SELECT DISTINCT event_type FROM product_events;"
with psycopg.connect(conn_str) as conn:
    print(pd.read_sql_query(diagnostic_query, conn))


         event_type
0       add_to_cart
1          checkout
2         page_view
3      product_view
4          purchase
5  remove_from_cart


/var/folders/7n/1gz7py493ldd6nlskw8mzbdc0000gn/T/ipykernel_72580/1212289705.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  print(pd.read_sql_query(diagnostic_query, conn))


In [14]:
sql_query_2_patched = """
SELECT product_id, SUM(revenue) as total_revenue
FROM product_events
WHERE event_type = 'purchase'
GROUP BY product_id
ORDER BY total_revenue DESC
LIMIT 5;
"""

with psycopg.connect(conn_str) as conn:
    df_sql_result = pd.read_sql_query(sql_query_2_patched, conn)

print("SQL Query 2 Patched Output (Top 5 Products):")
print(df_sql_result)


SQL Query 2 Patched Output (Top 5 Products):
  product_id  total_revenue
0     p_0071       36803.70
1     p_0067       36517.51
2     p_0061       35308.71
3     p_0062       34995.29
4     p_0059       34511.68


/var/folders/7n/1gz7py493ldd6nlskw8mzbdc0000gn/T/ipykernel_72580/2969968173.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sql_result = pd.read_sql_query(sql_query_2_patched, conn)


In [15]:
sql_query_4 = """
SELECT event_type, COUNT(*) as interaction_count
FROM product_events
WHERE event_type IN ('page_view', 'purchase')
GROUP BY event_type
ORDER BY interaction_count DESC;
"""

with psycopg.connect(conn_str) as conn:
    df_sql_result = pd.read_sql_query(sql_query_4, conn)

print("SQL Query 4 Output (Funnel Volume Counts):")
print(df_sql_result)


SQL Query 4 Output (Funnel Volume Counts):
  event_type  interaction_count
0  page_view               5991
1   purchase               2431


/var/folders/7n/1gz7py493ldd6nlskw8mzbdc0000gn/T/ipykernel_72580/938147244.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sql_result = pd.read_sql_query(sql_query_4, conn)


In [16]:
sql_query_5 = """
SELECT DISTINCT user_id, revenue, country
FROM product_events
WHERE event_type = 'purchase' AND revenue > 150.00
ORDER BY revenue DESC
LIMIT 5;
"""

with psycopg.connect(conn_str) as conn:
    df_sql_result = pd.read_sql_query(sql_query_5, conn)

print("SQL Query 5 Output (High-Value Purchase Sessions):")
print(df_sql_result)


SQL Query 5 Output (High-Value Purchase Sessions):
   user_id  revenue country
0  u_00103   1492.8      AU
1  u_00124   1492.8      CA
2  u_00223   1492.8      BR
3  u_00300   1492.8      DE
4  u_00317   1492.8      DE


/var/folders/7n/1gz7py493ldd6nlskw8mzbdc0000gn/T/ipykernel_72580/526713812.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sql_result = pd.read_sql_query(sql_query_5, conn)
